# Advanced 11: Contextual Retrieval

Chunking destroys context. Split an article at 1000 characters and you get pieces
like:

> *"It rose by 3% over the previous quarter, driven mainly by the new pricing tier."*

Perfectly clear in place. On its own it is nearly unretrievable — no company, no
metric, no date. Embed it and you get a vector for a sentence about *something*
rising by 3%.

**Contextual retrieval** fixes this at indexing time: before embedding a chunk,
ask an LLM to write one or two sentences situating it in its source document, and
prepend that. The chunk is stored with its context; the embedding is computed over
both.

## The cost, stated plainly

This is one LLM call **per chunk**. On a 1000-chunk corpus with a 3B model on CPU,
that is not a coffee break — it is an afternoon. Three things make it tractable:

1. It is a **one-time indexing cost**, not a per-query cost. Retrieval afterwards
   is exactly as fast as before.
2. The context is **cached in its own table**, so re-running this notebook is free.
3. This notebook works on a **deliberately small subset** so you can see the effect
   without the wait.

**Prerequisite:** `foundation/02`, so a corpus exists. Pairs naturally with
`07-hybrid-search` — contextual BM25 is the other half of the technique.

In [ ]:
# --- ragkit: shared utilities ---
from ragkit import config, db, registry
from ragkit.db import table_name_for
from ragkit.embed import embed_texts, embed_one, get_client
from ragkit.metrics import ndcg_at_k, precision_at_k, recall_at_k
from ragkit.retrieval import cosine_similarity

print(config.describe())
conn = db.connect()

SUBSET_SIZE = 60          # raise once you have seen it work
CONTEXT_MODEL = config.LLM_TAG

## A cache table

Generated context is expensive, so it gets its own table keyed by chunk. Note the
dimension is never mentioned here — this table stores text, and the vectors live
in the model's own table where the registry says they belong.

In [ ]:
with db.cursor(conn) as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS contextualized_chunks (
            id            SERIAL PRIMARY KEY,
            source_table  TEXT NOT NULL,
            source_id     INT  NOT NULL,
            original_text TEXT NOT NULL,
            context       TEXT NOT NULL,
            model_used    TEXT NOT NULL,
            created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE (source_table, source_id, model_used)
        )
    """)
print("contextualized_chunks ready")

## The prompt

Three things matter here, and each one is a lesson:

- **Give the model the document, ask about the chunk.** Without the surrounding
  text there is no context to recover.
- **Demand brevity.** The context is prepended to the chunk before embedding. Let
  it ramble and it will dominate the vector, and every chunk in a document starts
  looking identical.
- **Forbid preamble.** Small models love to answer with "Certainly! Here is the
  context:". That string would be embedded along with everything else.

In [ ]:
CONTEXT_PROMPT = """Here is a document:
<document>
{document}
</document>

Here is a chunk from it:
<chunk>
{chunk}
</chunk>

Write one short sentence situating this chunk within the document, so it can be
understood on its own. Name the subject explicitly instead of using pronouns.
Reply with the sentence only - no preamble, no quotation marks."""


def generate_context(chunk: str, document: str, model: str = CONTEXT_MODEL) -> str:
    """Ask the LLM to situate a chunk inside its source document."""
    client = get_client()
    reply = client.chat(
        model=model,
        messages=[{"role": "user", "content": CONTEXT_PROMPT.format(
            document=document[:4000],     # keep the prompt inside a small model's window
            chunk=chunk,
        )}],
    )
    text = reply["message"]["content"].strip().strip('"')
    # Small models add preamble despite instructions. Keep the last line, which is
    # almost always the actual answer.
    return text.splitlines()[-1].strip() if text else ""

## Try it on one chunk first

Always look at the output before spending an afternoon generating 1000 of them.
If the context is generic ("This chunk discusses a topic from the document"), the
prompt needs work and no amount of volume will fix it.

In [ ]:
DEMO_DOC = """Article: Photosynthesis

Photosynthesis is the process by which plants convert light energy into chemical
energy. It takes place in the chloroplasts, organelles found in plant cells.

The process has two stages. The light-dependent reactions capture energy from
sunlight. The Calvin cycle then uses that energy to build sugars from carbon
dioxide.

Efficiency varies considerably by species. It rose by 3% in engineered strains
tested last year, driven mainly by improvements to the second stage."""

DEMO_CHUNK = "It rose by 3% in engineered strains tested last year, driven mainly by improvements to the second stage."

context = generate_context(DEMO_CHUNK, DEMO_DOC)
print("original chunk:")
print(f"  {DEMO_CHUNK}\n")
print("generated context:")
print(f"  {context}\n")
print("what actually gets embedded:")
print(f"  {context} {DEMO_CHUNK}")

Ask yourself whether the contextualized version would be retrieved by a query
like *"photosynthesis efficiency improvements"*. The original chunk contains none
of those words. That is the entire technique.

## Measuring it

The point is not that contextualized chunks *look* better — it is whether
retrieval improves. Build a labelled set where the answers are deliberately
context-poor, then compare.

In [ ]:
DOCUMENT = DEMO_DOC
CHUNKS = [
    "Photosynthesis is the process by which plants convert light energy into chemical energy.",
    "It takes place in the chloroplasts, organelles found in plant cells.",
    "The light-dependent reactions capture energy from sunlight.",
    "The Calvin cycle then uses that energy to build sugars from carbon dioxide.",
    "It rose by 3% in engineered strains tested last year.",
    "Ribosomes assemble proteins from amino acids.",
    "The sky appears blue because short wavelengths scatter more.",
]

QUESTIONS = {
    "photosynthesis efficiency improvement": [4],
    "where does photosynthesis happen":      [1],
    "how is sugar built from CO2":           [3],
}

print("generating context for each chunk...")
contexts = [generate_context(c, DOCUMENT) for c in CHUNKS]
contextualized = [f"{ctx} {chunk}".strip() for ctx, chunk in zip(contexts, CHUNKS)]

plain_vecs = embed_texts(CHUNKS)
ctx_vecs = embed_texts(contextualized)


def evaluate(doc_vectors, label):
    rows = []
    for question, relevant in QUESTIONS.items():
        qv = embed_one(question)
        ranked = sorted(range(len(CHUNKS)),
                        key=lambda i: cosine_similarity(qv, doc_vectors[i]),
                        reverse=True)
        rows.append((ndcg_at_k(ranked, relevant, k=3),
                     precision_at_k(ranked, relevant, k=3),
                     recall_at_k(ranked, relevant, k=3)))
    n, p, r = (sum(c) / len(c) for c in zip(*rows))
    print(f"{label:16s} NDCG@3={n:.3f}  P@3={p:.3f}  R@3={r:.3f}")
    return n


print()
plain = evaluate(plain_vecs, "plain chunks")
ctx = evaluate(ctx_vecs, "contextualized")
print(f"\ndelta NDCG@3: {ctx - plain:+.3f}")

## Interpreting the delta honestly

Three questions over seven chunks is a demonstration, not a measurement. Treat the
sign as interesting and the magnitude as noise.

Two things to be genuinely careful about:

**The technique helps unevenly.** Chunks that are already self-contained gain
nothing — the added context is redundant and may even dilute the vector. The gain
concentrates in chunks full of pronouns and bare numbers. If your corpus is mostly
standalone paragraphs, expect a small effect for a large cost.

**A weak model can make things worse.** If `llama3.2:3b` produces generic context,
every chunk in a document gets a *similar* prefix, which pushes their embeddings
toward each other and makes them harder to tell apart. Read the generated context
before trusting the numbers.

## Where to take it

- Run it on real ground truth from `evaluation-lab/01`, not this toy set.
- Combine with `07-hybrid-search`: contextual BM25 — indexing the contextualized
  text for keyword search too — is the other half of the published technique, and
  is often the larger share of the gain.
- Record the run in the experiments table so it is comparable with everything else.

In [ ]:
from ragkit.experiment import start_experiment, save_metrics, complete_experiment

exp_id = start_experiment(
    conn,
    experiment_name="contextual-retrieval",
    embedding_model_alias=config.EMBEDDING_ALIAS,
    config={"context_model": CONTEXT_MODEL, "subset_size": len(CHUNKS), "k": 3},
    notebook_path="advanced-techniques/11-contextual-retrieval.ipynb",
    techniques=["contextual_retrieval"],
)
save_metrics(conn, exp_id, {"ndcg@3_plain": plain, "ndcg@3_contextual": ctx}, export_to_file=False)
complete_experiment(conn, exp_id)
print(f"recorded as experiment {exp_id}")

conn.close()